In [86]:
import pandas as pd
from Bio import SeqIO
import sys, re
import os,glob
thresholds = [97,98,99,99.5]
segments = ['PB2','PB1','PA','HA','NP','NA','MP','NS']
blast = pd.read_csv("h5n1_geno_refs_sept24.out",sep="\t")
blast.columns = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore']

In [82]:
import json
blast[['query_isolate_name', 'query_genotype','query_subtype','query_segment']] = blast['qseqid'].str.split('|', expand=True)
blast[['hit_isolate_name', 'hit_genotype','hit_subtype','hit_segment']] = blast['sseqid'].str.split('|', expand=True)
blast.to_csv("blast_h5n1_geno_refs_sept24.csv")
queries = list(set(blast['query_isolate_name']))

for t in thresholds:
    subblast = blast[blast['pident']>=t]
    subblast = subblast[subblast['pident']>=t]
    for s in segments:
        segblast = subblast[subblast['query_segment']==s]
        groupdict = {}
        groups = 1
        
        for n,q in enumerate(queries):
            hitblast = segblast[segblast['query_isolate_name']==q]
            if hitblast.shape[0]==0:
                groupdict[f'{s}_group{groups}'] = q
                groups = groups+1
            else:
            #    if n ==0:
            #        
            #        grouplist = list(hitblast['hit_isolate_name'])
             #       grouplist.append(q)          
             #       groupdict[f'{s}_group{groups}'] = list(set(grouplist))
             #       groups = groups+1
             #   else:
                    ingroup = False
                    #check if query is already in a group? 
                    for key, val in groupdict.items():
                        #print(key, val)
                        if q in list(val):
                           # print("{} : {}".format(key, q))
                            ingroup = True
                            continue
                    if not ingroup:
                        newgrouplist = []
                        
                        grouplist = list(hitblast['hit_isolate_name'])
                        grouplist.append(q) 
                        for g in grouplist:
                            ng = False
                            for key, val in groupdict.items():
                            #print(key, val)
                                if g in list(val):
                                   # print("{} : {}".format(key, q))
                                    ng = True
                                    continue   
                            if not ng:
                                 newgrouplist.append(g)
                        groupdict[f'{s}_group{groups}'] = list(set(newgrouplist))
                        ingroup = True
                        groups = groups+1
                        pass
       # print(groupdict.keys())
    #    grouptab = pd.DataFrame.from_dict(groupdict, orient='index')
     #   grouptab.columns = ['sequences']

        with open(f'blast_geno_threshold{t}_{s}.json', 'w') as fp:
            json.dump(groupdict, fp)
        #grouptab.to_csv(.csv')


PB2_group1 : B5.1_ref
PB2_group2 : A/bald_eagle/Florida/W22-189/2022
PB2_group2 : B2.1_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group2 : B1.3_ref
PB2_group1 : A/avian/Burkina_Faso/21VIR11911-3/2021
PB2_group1 : A/turkey/England/016515/2022
PB2_group4 : A/chicken/Scotland/054477/2021
PB2_group1 : Minor09_ref
PB2_group1 : A3_ref
PB2_group8 : Minor18_ref
PB2_group2 : Minor29_ref
PB2_group1 : A/peregrine_falcon/Ireland/000191_22VIR1325-15/2022
PB2_group1 : A2_ref
PB2_group1 : A/Eurasian_Wigeon/Netherlands/1/2020
PB2_group2 : Minor26_ref
PB2_group8 : B3.5_ref
PB2_group2 : Minor13_ref
PB2_group2 : B2.2_ref
PB2_group1 : Minor12_ref
PB2_group2 : Minor19_ref
PB2_group2 : Minor31_ref
PB2_group4 : A/chicken/England/152082/2022
PB2_group2 : Minor30_ref
PB2_group15 : A/mute_swan/Wales/048068/2020
PB2_group3 : Minor32_ref
PB2_group3 : B3.2_ref
PB2_group1 : A1

In [101]:
from collections import Counter
# we want the different combinations / constellations in a table

#blast[['query_isolate_name', 'query_genotype','query_subtype','query_segment']] = blast['qseqid'].str.split('|', expand=True)
#blast[['hit_isolate_name', 'hit_genotype','hit_subtype','hit_segment']] = blast['sseqid'].str.split('|', expand=True)
blast[['query_isolate_name', 'query_genotype','query_subtype','query_segment']] = blast['qseqid'].str.split('|', expand=True)
blast[['hit_isolate_name', 'hit_genotype','hit_subtype','hit_segment']] = blast['sseqid'].str.split('|', expand=True)
queries = list(set(blast['query_isolate_name']))
tabcols = ['sequence','genotype','subtype']
tabcols.extend(segments)
print(tabcols)

for t in thresholds: 
    testdf = []
    for q in queries:
        hitblast = blast[blast['query_isolate_name']==q]
        seqinfo = [q,hitblast['query_genotype'].iloc[0],hitblast['query_subtype'].iloc[0]]
      #  print(t,q)
        for s in segments:
            
            with open(f'blast_geno_threshold{t}_{s}.json', 'r') as file:
                data = file.read()
            tsdict = json.loads(data)
          #  print(tsdict)
            found = False
            if not found:
                for key, val in tsdict.items():
                    #  print(key)
                        if q in list(val):
                            seqinfo.append(key)
                            found = True
                            exit
        if len(seqinfo)>11:
            print(seqinfo)
        testdf.append(seqinfo)
    thresholddf = pd.DataFrame(testdf, columns = tabcols)
    for s in segments:
        thresholddf[s] = thresholddf[s].fillna("Absent") 
    print(thresholddf.head)
    thresholddf["constellation"] = thresholddf[['PB2','PB1','PA','HA','NP','NA','MP','NS']].agg("|".join, axis=1)
  #df[["Courses", "Duration"]].apply(lambda x: " ".join(x), axis =1)
    print(Counter(thresholddf['constellation']))
    thresholddf.to_csv(f'blast_geno_threshold_table{t}.csv')

['sequence', 'genotype', 'subtype', 'PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS']
<bound method NDFrame.head of                           sequence    genotype subtype          PB2  \
0                           A5_ref          A5    H5N1   PB2_group1   
1                      Minor28_ref     Minor28    H5N1   PB2_group2   
2                         B3.1_ref        B3.1    H5N1   PB2_group3   
3                         B5.1_ref        B5.1    H5N1   PB2_group1   
4    A/Chicken/England/005435/2024    DL_AIV01    H5N1   PB2_group4   
..                             ...         ...     ...          ...   
84                     Minor10_ref     Minor10    H5N1   PB2_group2   
85     A/turkey/Tyumen/15-14V/2021         G03    H5N1    HA_group1   
86                        B1.1_ref        B1.1    H5N1   PB2_group2   
87  A/Wild_bird/Korea/K22-742/2022  Genotype_I    H5N1  PB2_group20   
88                     Minor08_ref     Minor08    H5N1  PB2_group14   

            PB1          PA    